# EEG_39 — Preprocessing XDF → CSV (Francesco dataset)

Pipeline: XDF grezzo (512 Hz, 62 ch) → filtro bandpass + notch → resample 256 Hz → CAR → epoching → CSV

Output per sessione (merge nella cartella esistente di Paolo sul server):
- `PXXX_SYYY/parola_read.csv`  — lettura visiva (61, 384)  ← NUOVO
- `PXXX_SYYY/riposo_NNN.csv`   — riposo 1.5s (61, 384)     ← NUOVO
- `PXXX_SYYY/parola_img.csv`   — SALTATO (già in Paolo)

Rsync al server dopo ogni soggetto (incrementale, resume-safe).

In [ ]:
# ============================================================
# CONFIG
# ============================================================
from pathlib import Path

XDF_ROOT = Path("/Users/danieleuras/Library/CloudStorage/OneDrive-PolitecnicodiMilano/File di Francesco Iacomi - Subjects_xdf")

# Output locale — stessa struttura di Paolo, viene mergiato sul server
CSV_OUT = Path("/Users/danieleuras/Documents/GitHub/miralis-hypergraph-imagined-speech/data/raw_csv_new/training_set")

# Destinazione server — cartella esistente di Paolo
SERVER_DEST = 'spinlabs01:/home/daniele_u/miralis-hypergraph-imagined-speech/data/raw_csv/training_set/'

# Preprocessing
L_FREQ     = 1.0
H_FREQ     = 100.0
NOTCH_FREQ = 50.0
SFREQ_IN   = 512
SFREQ_OUT  = 256
N_CHAN     = 61
EPOCH_DUR  = 1.5
N_SAMP     = int(EPOCH_DUR * SFREQ_OUT)  # 384

# Soggetti già processati da Paolo → skip _img (inutile ricalcolare)
PAOLO_SUBJECTS = set(range(0, 74))  # P000–P073

# Processa SOLO questi soggetti (None = tutti). Lo skip locale non serve:
# i CSV vengono rsyncati sul server e cancellati in locale, quindi filtra qui.
ONLY_SUBJECTS = None   # {3,59,60,90} erano i buchi, ora risolti

RSYNC_EACH_SUBJECT = True

print(f'N_SAMP={N_SAMP}  PAOLO_SUBJECTS=P000–P073  RSYNC_EACH={RSYNC_EACH_SUBJECT}')

In [6]:
import numpy as np
import pandas as pd
import mne
import pyxdf
import subprocess
import logging
import time
from pathlib import Path
from tqdm.auto import tqdm
from collections import defaultdict

mne.set_log_level('WARNING')
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)-8s %(message)s', datefmt='%H:%M:%S')
log = logging.getLogger('eeg39')
print('Import OK')

Import OK


In [7]:
# ============================================================
# FUNZIONI
# ============================================================

def xdf_to_outdir(xdf_path: Path, csv_out: Path) -> Path:
    """sub-P000/ses-S001/.../file.xdf  →  csv_out/P000_S001/"""
    parts = xdf_path.parts
    subj  = next(p for p in parts if p.startswith('sub-P'))
    sess  = next(p for p in parts if p.startswith('ses-S'))
    return csv_out / f'{subj.replace("sub-", "")}_{sess.replace("ses-", "")}'


def process_session(xdf_path: Path, out_dir: Path, skip_img: bool = True) -> dict:
    """
    Processa un file XDF. Genera _read.csv e riposo_NNN.csv.
    Se skip_img=True salta i _img.csv.
    Skip automatico se out_dir contiene già *_read.csv.
    """
    stats = {'n_read': 0, 'n_img': 0, 'n_riposo': 0, 'n_errors': 0, 'skipped': False}

    # Resume: salta se già processato
    if out_dir.exists() and any(out_dir.glob('*_read.csv')):
        stats['skipped'] = True
        return stats

    # Carica XDF
    try:
        streams, _ = pyxdf.load_xdf(str(xdf_path))
    except Exception as e:
        log.error(f'Caricamento fallito {xdf_path.name}: {e}')
        stats['n_errors'] += 1
        return stats

    eeg_stream = next((s for s in streams if s['info']['type'][0].upper() == 'EEG'), None)
    mrk_stream = next((s for s in streams if s['info']['type'][0] == 'Markers'), None)
    if eeg_stream is None or mrk_stream is None:
        log.error(f'Stream mancante in {xdf_path.name}')
        stats['n_errors'] += 1
        return stats

    # EEG raw
    raw_data = np.array(eeg_stream['time_series'], dtype=np.float32).T  # (62, n)
    sfreq_in = float(eeg_stream['info']['nominal_srate'][0])
    t_eeg_start = float(eeg_stream['time_stamps'][0])
    raw_data = raw_data[:N_CHAN, :]  # rimuovi ultimo canale

    try:
        ch_info  = eeg_stream['info']['desc'][0]['channel']
        ch_names = [ch['name'][0] for ch in ch_info[:N_CHAN]]
    except Exception:
        ch_names = [f'EEG{i:03d}' for i in range(N_CHAN)]

    # Preprocessing MNE
    info = mne.create_info(ch_names=ch_names, sfreq=sfreq_in, ch_types='eeg')
    raw  = mne.io.RawArray(raw_data * 1e-6, info, verbose=False)
    raw.filter(L_FREQ, H_FREQ, method='iir', verbose=False)
    raw.notch_filter(NOTCH_FREQ, verbose=False)
    if sfreq_in != SFREQ_OUT:
        raw.resample(SFREQ_OUT, verbose=False)
    raw.set_eeg_reference('average', verbose=False)

    proc = raw.get_data() * 1e6  # (61, n_resampled)
    n_resampled = proc.shape[1]

    mrk_ts  = mrk_stream['time_stamps']
    mrk_val = [m[0] for m in mrk_stream['time_series']]

    out_dir.mkdir(parents=True, exist_ok=True)

    def save_epoch(marker_time: float, filename: str) -> bool:
        rel = marker_time - t_eeg_start
        idx = int(round(rel * SFREQ_OUT))
        if idx < 0 or idx + N_SAMP > n_resampled:
            return False
        epoch = proc[:, idx:idx + N_SAMP].astype(np.float32)
        pd.DataFrame(epoch).to_csv(out_dir / filename, header=False, index=False, float_format='%.6f')
        return True

    marker_pairs = list(zip(mrk_ts, mrk_val))
    riposo_count = 0

    for i, (ts, label) in enumerate(marker_pairs):
        if '_img' in label:
            if not skip_img:
                word = label.replace('_img', '')
                ok = save_epoch(ts, f'{word}_img.csv')
                stats['n_img'] += int(ok)
                stats['n_errors'] += int(not ok)

        elif '_read' in label:
            word = label.replace('_read', '')
            ok = save_epoch(ts, f'{word}_read.csv')
            stats['n_read'] += int(ok)
            stats['n_errors'] += int(not ok)

        elif label == 'riposo':
            rest_end = marker_pairs[i + 1][0] if i + 1 < len(marker_pairs) else ts + 20.0
            n_windows = int((rest_end - ts) // EPOCH_DUR)
            for k in range(n_windows):
                riposo_count += 1
                ok = save_epoch(ts + k * EPOCH_DUR, f'riposo_{riposo_count:03d}.csv')
                stats['n_riposo'] += int(ok)
                stats['n_errors'] += int(not ok)

    return stats


def rsync_subject(local_subj_dir: Path, server_dest: str):
    """Rsync una singola cartella soggetto al server (merge, non sovrascrive _img di Paolo)."""
    cmd = [
        'rsync', '-az',
        '--ignore-existing',
        str(local_subj_dir) + '/',
        server_dest + local_subj_dir.name + '/'
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        log.error(f'rsync fallito per {local_subj_dir.name}: {result.stderr.strip()}')
        return False
    return True


print('Funzioni OK')

Funzioni OK


In [8]:
# ============================================================
# TEST SU UN SINGOLO FILE
# ============================================================
test_xdf = XDF_ROOT / 'sub-P000' / 'ses-S001' / 'eeg' / 'sub-P000_ses-S001_task-Default_run-001_eeg.xdf'
test_out = CSV_OUT / 'P000_S001_test'
test_out.mkdir(parents=True, exist_ok=True)

# P000 è in Paolo → skip_img=True
skip_img_test = 0 in PAOLO_SUBJECTS

t0 = time.time()
stats = process_session(test_xdf, test_out, skip_img=skip_img_test)
print(f'Tempo: {time.time()-t0:.1f}s')
print(f'read={stats["n_read"]}  img={stats["n_img"]}  riposo={stats["n_riposo"]}  errori={stats["n_errors"]}')

for f in sorted(test_out.iterdir())[:6]:
    arr = pd.read_csv(f, header=None).values
    print(f'  {f.name}: {arr.shape} {arr.dtype}')

Tempo: 0.0s
read=0  img=0  riposo=0  errori=0
  accendere_read.csv: (61, 384) float64
  acqua_read.csv: (61, 384) float64
  adesso_read.csv: (61, 384) float64
  aiuto_read.csv: (61, 384) float64
  amore_read.csv: (61, 384) float64
  anche_read.csv: (61, 384) float64


In [9]:
# ============================================================
# SCOPERTA FILE E RAGGRUPPAMENTO PER SOGGETTO
# ============================================================
from itertools import groupby

all_xdf = sorted(XDF_ROOT.rglob('*.xdf'))

# Raggruppa per soggetto (sub-PXXX)
def get_subject(xdf_path):
    return next(p for p in xdf_path.parts if p.startswith('sub-P'))

subjects = {}
for xdf in all_xdf:
    subj = get_subject(xdf)
    subjects.setdefault(subj, []).append(xdf)

print(f'Soggetti: {len(subjects)}')
print(f'File XDF totali: {len(all_xdf)}')

# Quanti già processati (hanno *_read.csv)
done = sum(
    1 for xdf in all_xdf
    if any(xdf_to_outdir(xdf, CSV_OUT).glob('*_read.csv'))
)
print(f'Sessioni già processate: {done} / {len(all_xdf)}')

Soggetti: 91
File XDF totali: 446
Sessioni già processate: 0 / 446


In [ ]:
# ============================================================
# LOOP PRINCIPALE — preproc → csv → rsync → delete locale
# ============================================================
import shutil

total   = defaultdict(int)
failed  = []
t_all   = time.time()

if ONLY_SUBJECTS is not None:
    # glob diretto per-soggetto: forza l'enumerazione OneDrive (le cartelle
    # cloud-only non vengono viste dal rglob globale -> soggetti 'spariti')
    _items = []
    for _sid in sorted(ONLY_SUBJECTS):
        _xs = sorted((XDF_ROOT / f'sub-P{_sid:03d}').rglob('*.xdf'))
        if not _xs:
            log.warning(f'sub-P{_sid:03d}: nessun XDF trovato (cloud-only/non scaricato?)')
        _items.append((f'sub-P{_sid:03d}', _xs))
else:
    _items = list(subjects.items())


for subj_name, xdf_list in tqdm(_items, desc='Soggetti'):
    subj_num = int(subj_name.replace('sub-P', ''))
    skip_img = subj_num in PAOLO_SUBJECTS  # True per P000-P073, False per P074-P090

    subj_dirs_local = []

    for xdf_path in sorted(xdf_list):
        out_dir = xdf_to_outdir(xdf_path, CSV_OUT)
        try:
            s = process_session(xdf_path, out_dir, skip_img=skip_img)
            for k, v in s.items():
                if isinstance(v, int):
                    total[k] += v
            if s['n_errors'] > 0:
                failed.append(xdf_path.name)
            if not s['skipped']:
                subj_dirs_local.append(out_dir)
        except Exception as e:
            log.error(f'FAIL {xdf_path.name}: {e}')
            failed.append(xdf_path.name)

    # Rsync dopo ogni soggetto → poi cancella locale se OK
    if RSYNC_EACH_SUBJECT and subj_dirs_local:
        for local_dir in subj_dirs_local:
            ok = rsync_subject(local_dir, SERVER_DEST)
            if ok:
                shutil.rmtree(local_dir)
                log.info(f'  rsync OK + cancellato locale → {local_dir.name}')
            else:
                log.warning(f'  rsync FALLITO, locale conservato → {local_dir.name}')

elapsed = time.time() - t_all
print(f'\n=== DONE in {elapsed/60:.1f} min ===')
print(f'read={total["n_read"]}  img={total["n_img"]}  riposo={total["n_riposo"]}  errori={total["n_errors"]}')
if failed:
    print(f'\nFalliti ({len(failed)}): {failed[:10]}')